# DBR(동아비즈니스리뷰) 기사 본문 크롤링

- 사이트: https://dbr.donga.com
- 수집 항목: 기사 본문 (식별용 URL 함께 저장)
- 도구: Selenium + BeautifulSoup
- 저장: CSV

## 1. 라이브러리 불러오기

In [1]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from bs4 import BeautifulSoup
import pandas as pd
import time
import re
import os
from pathlib import Path

## 2. Selenium 드라이버 설정 및 사이트 접속

In [2]:
# 크롬 옵션 설정
options = Options()
options.add_argument('--start-maximized')
options.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36')

# 드라이버 실행
driver = webdriver.Chrome(options=options)

# DBR 메인 페이지 접속
url = 'https://dbr.donga.com'
driver.get(url)
time.sleep(2)

## 3. 기사 링크 수집
- 메인 페이지에서 `article_no` 가 포함된 링크를 수집한다.
- 같은 기사가 다른 경로로 중복 수집되므로 `article_no` 값으로 중복 제거한다.

In [3]:
soup = BeautifulSoup(driver.page_source, 'html.parser')

links = []
seen_nos = set()
for a in soup.select('a[href*="article_no"]'):
    href = a.get('href')
    if not href:
        continue
    if href.startswith('/'):
        href = 'https://dbr.donga.com' + href
    m = re.search(r'article_no/(\d+)', href)
    if not m:
        continue
    no = m.group(1)
    if no in seen_nos:
        continue
    seen_nos.add(no)
    links.append(href)

print(f'중복 제거 후 기사 링크 수: {len(links)}')
links[:5]

중복 제거 후 기사 링크 수: 16


['https://dbr.donga.com/article/view/1303/article_no/12026',
 'https://dbr.donga.com/article/view/1101/article_no/12106',
 'https://dbr.donga.com/article/view/1901/article_no/12111',
 'https://dbr.donga.com/article/view/1101/article_no/12109/ac/m_best',
 'https://dbr.donga.com/article/view/1201/article_no/12088/ac/m_best']

## 4. 기사 상세 페이지 본문 파싱 함수
- 본문: `div.cont-article` (이미지/스크립트 제거 후 텍스트만 추출)

In [4]:
def parse_article(driver, url):
    driver.get(url)
    time.sleep(1.5)
    soup = BeautifulSoup(driver.page_source, 'html.parser')

    # 기사 제목 추출 (section명 / 기사 제목)
    title = ''
    header = soup.select_one('div.header-cont')
    if header:
        parts = []
        sub = header.select_one('p.subtitle')
        h4  = header.select_one('h4.title')
        if sub: parts.append(sub.get_text(strip=True))
        if h4:  parts.append(h4.get_text(strip=True))
        title = ' / '.join(parts)

    # 본문 영역
    body_el = soup.select_one('div.cont-article')
    if not body_el:
        return {'title': title, 'body': '', 'url': url}

    # 노이즈 제거 (페이월 CTA, 저자 정보, 저작권, 네비게이션, 인기기사)
    for sel in [
        'section.preview',       # 멤버십 가입 CTA
        'ul.new_author_wrap',    # 저자 이메일/약력
        'div.copyright_notice',  # 저작권 고지
        'div.ta-r',              # 이전/목록/다음
        'ul.relate_article',     # 인기기사 목록
    ]:
        for el in body_el.select(sel):
            el.decompose()
    # "인기기사" 텍스트만 있는 요소 제거
    for el in list(body_el.children):
        if hasattr(el, 'get_text') and el.get_text(strip=True) == '인기기사':
            el.decompose()
    # script/style 제거
    for tag in body_el.find_all(['script', 'style']):
        tag.decompose()

    body = body_el.get_text('\n', strip=True)

    return {'title': title, 'body': body, 'url': url}

## 5. 링크 순회하며 데이터 수집

In [5]:
articles = []

for i, link in enumerate(links, 1):
    try:
        data = parse_article(driver, link)
        articles.append(data)
        print(f'[{i}/{len(links)}] {data["title"][:40]} | 본문 {len(data["body"])}자')
    except Exception as e:
        print(f'[{i}/{len(links)}] 실패: {e}')
        continue

print(f'\n총 {len(articles)}건 수집 완료')

[1/16] DBR ‘찐팬’ 5인의 축사 / “DBR, 기술 변화 속 길을 제시하는  | 본문 1069자


[2/16] Special Report / 공급망 효율의 종말과 기업 생존 전략 | 본문 1855자


[3/16] DBR Case Study / 日 잡화점 돈키호테, 36년 연속 매출·이 | 본문 1595자


[4/16] Special Report / 중동發 에너지 위기와 ESG 전략 재편 | 본문 1354자


[5/16] Innovation / ‘성과주의·포용성’ 이중적 조직문화혁신 실행 단계 | 본문 1525자


...

[12/16] Interview: 이한대 싸이더스 대표 · 최윤호 싸이더스 베트남법인  | 본문 1970자


[13/16] Special Report / 조직 생산성 가르는 직원 돌봄 부담 | 본문 1697자


[14/16] Special Report / 한국 스타트업 창업자들이 말하는 미국 진출 | 본문 15004자


[15/16] Organizational Behavior / 자기 생각을 성찰하는 ‘메 | 본문 2411자


[16/16] 두쫀쿠 열풍으로 본 유행 읽기 / 기업 진입하면 이미 끝나는 단기 유행열 | 본문 7566자

총 16건 수집 완료


## 6. 드라이버 종료

In [6]:
driver.quit()

## 7. CSV 저장

In [7]:
save_dir = Path(os.path.abspath(''))
save_path = save_dir / 'DBR_기사본문.csv'

df = pd.DataFrame(articles, columns=['title', 'body', 'url'])
df.to_csv(save_path, index=False, encoding='utf-8-sig')
print(f'저장 완료: {save_path}')
df.head()

저장 완료: C:\Users\smhrd\Documents\Claude\Projects\실전프로젝트\크롤링\DBR_기사본문.csv


,title,body,url
0,"DBR ‘찐팬’ 5인의 축사 / “DBR, 기술 변화 속 길을 제시하는 지적 나침반”","함께 만들고, 함께 읽고 또 널리 전해 주신 필자와 독자 여러분의 성원 덕분에 DB...",https://dbr.donga.com/article/view/1303/articl...
1,Special Report / 공급망 효율의 종말과 기업 생존 전략,‘최고 효율’ 아닌 ‘최다 대안’ 가져야\n북미·亞 등 병렬 생산 역량 절실\nAr...,https://dbr.donga.com/article/view/1101/articl...
2,"DBR Case Study / 日 잡화점 돈키호테, 36년 연속 매출·이익 성장 비결",재고 상품·심야 시간 ‘사각지대’를 수익화\n역발상 ‘혼돈의 진열’로 발견의 재미 ...,https://dbr.donga.com/article/view/1901/articl...
3,Special Report / 중동發 에너지 위기와 ESG 전략 재편,이젠 ‘에너지 생산성’이 기업 핵심 경쟁력\n복원력 강한 전력 포트폴리오로 전환을\...,https://dbr.donga.com/article/view/1101/articl...
4,Innovation / ‘성과주의·포용성’ 이중적 조직문화혁신 실행 단계에선 득보다...,▶ Based on “The duality of duality: Generative...,https://dbr.donga.com/article/view/1201/articl...
